In [ ]:
import os
import re
import pickle as pkl
import ast
import json
from collections import Counter
from typing import List, Tuple, Any, Optional
import time
import random

import pandas as pd
import numpy as np
import calibration as cal

import torch

from sklearn.model_selection import train_test_split
from sklearn.calibration import calibration_curve
from sklearn.metrics import brier_score_loss, roc_auc_score

from tqdm.auto import tqdm
from tenacity import retry, wait_random_exponential, stop_after_attempt, retry_if_exception_type

from pydantic import BaseModel, Field, confloat

import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8')
pal = plt.rcParams['axes.prop_cycle'].by_key()['color']

import asyncio
import nest_asyncio
import openai
from openai import OpenAI, AsyncOpenAI
from openai import RateLimitError, APITimeoutError, APIConnectionError, InternalServerError

print("openai package version:", getattr(openai, "__version__", "unknown"))

In [ ]:
api_key = ''

In [ ]:
from llm_unsupervised_conf.metrics import *
from llm_unsupervised_conf.math_utils import stack_embeddings
from llm_unsupervised_conf.utils import maybe_add_verbal_conf_row, load_out_df, set_seed
from llm_unsupervised_conf.calibration import fit_predict_prob_models
from llm_unsupervised_conf.plots import plot_reliability, plot_avg_and_worstcase_by_method, plot_method_comparisons

In [ ]:
# Prefer environment variable. You can also set `api_key = "..."` in a prior cell.
api_key = os.environ.get("OPENAI_API_KEY") or globals().get("api_key", None)
if not api_key:
    raise ValueError("Set OPENAI_API_KEY in your environment (or define `api_key` in a prior cell).")

# Sync and async clients (we manage retries ourselves)
client = OpenAI(api_key=api_key, timeout=60, max_retries=0)
aclient = AsyncOpenAI(api_key=api_key, timeout=60, max_retries=0)

# Jupyter runs an event loop already; this makes nested awaits usable from sync wrappers.
nest_asyncio.apply()


In [ ]:
# -----------------------------
# Fast Decision-Maker (DM) experiment runner (async + concurrency)
# -----------------------------

TEMPERATURE = 0.0

# Main speed knob: number of in-flight API calls
DM_CONCURRENCY = 30

# Checkpointing
SAVE_EVERY = 20000
RESULTS_PATH = "dm_results.parquet"

# Which provided confidence scores to compare
METHODS = ["vc", "lp", "ours"]

class DecisionMakerOutput(BaseModel):
    final_answer: str = Field(...)
    final_confidence: confloat(ge=0.0, le=1.0) = Field(...)

SYSTEM_PROMPT = """You are acting as a downstream decision maker being advised by an LLM that outputs an answer and its confidence score.

You will be given:
(1) a question,
(2) a candidate answer produced by an upstream system,
(3) a numeric confidence score in [0,1] from that upstream system.

CRITICAL INSTRUCTIONS:
- Treat the provided confidence score as probabilistically informative: interpret it as the probability that the candidate answer is correct.
- Do not ignore the candidate answer + the numeric score. Do NOT solve the problem from scratch without considering them.
- Still make your own assessment, and output a new answer and confidence score based on your knowledge.
- The final_answer string should be a short answer (e.g., the name of a single entity)

Output exactly:
- final_answer (string)
- final_confidence (number in [0,1])
"""

def build_user_prompt(question: str, candidate_answer: str, method_name: str, score: float) -> str:
    return (
        f"Question:\n{question}\n\n"
        f"Candidate answer:\n{candidate_answer}\n\n"
        f"Upstream confidence score '{method_name}' in [0,1]: {score}\n"
    )

def _clean_candidate_answer(ans: str) -> str:
    s = str(ans)
    # your existing cleaning; keep it conservative
    s = s.split("\\text{")[-1]
    s = s.replace("\\", "")
    return s.strip().lower()

# Retry only transient errors
TRANSIENT = (RateLimitError, APITimeoutError, APIConnectionError, InternalServerError)

async def call_decision_maker_async(
    question: str,
    candidate_answer: str,
    method_name: str,
    score: float,
    max_attempts: int = 8,
) -> DecisionMakerOutput:
    p = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": build_user_prompt(question, candidate_answer, method_name, score)},
    ]

    for attempt in range(max_attempts):
        try:
            if MODEL_NAME == "gpt-4o-mini":
                resp = await aclient.responses.parse(
                    model=MODEL_NAME,
                    input=p,
                    temperature=TEMPERATURE,
                    text_format=DecisionMakerOutput,
                )
            else:
                resp = await aclient.responses.parse(
                    model=MODEL_NAME,
                    input=p,
                    text_format=DecisionMakerOutput,
                )
            return resp.output_parsed
        except TRANSIENT:
            # exponential backoff + jitter
            base = 0.5 * (2 ** attempt)
            sleep = min(20.0, base) + random.random() * 0.2
            await asyncio.sleep(sleep)
        except Exception:
            # permanent error: raise immediately
            raise

    raise RuntimeError(f"DM call failed after {max_attempts} attempts.")

async def run_dm_experiment_async(
    df: pd.DataFrame,
    methods: List[str] = METHODS,
    results_path: str = RESULTS_PATH,
    max_rows: Optional[int] = None,
    concurrency: int = DM_CONCURRENCY,
    save_every: int = SAVE_EVERY,
) -> pd.DataFrame:
    """
    For each row and each method, ask the DM model to produce (final_answer, final_confidence)
    conditioned on the row's (question, candidate_answer, provided_score).

    Fast: runs with async concurrency; checkpointed; resumable.
    """

    needed_cols = {"id", "question", "answer", "correct", "ground_truth"}
    missing = needed_cols - set(df.columns)
    if missing:
        raise ValueError(f"DataFrame missing required columns: {missing}")

    for m in methods:
        if m not in df.columns:
            raise ValueError(f"DataFrame missing confidence column '{m}'")

    # Resumable: load existing results if present
    done = set()
    results = []
    # if os.path.exists(results_path):
    #     prev = pd.read_parquet(results_path)
    #     results = prev.to_dict("records")
    #     done = set(zip(prev["id"].astype(str), prev["method"]))
    #     print(f"Loaded {len(prev)} existing results from {results_path}.")

    work_df = df.copy()
    if max_rows is not None:
        work_df = work_df.iloc[:max_rows].copy()

    # Build job list
    jobs = []
    for _, row in work_df.iterrows():
        rid = str(row["id"])
        question = str(row["question"])
        candidate_answer = _clean_candidate_answer(row["answer"])
        correct = int(row["correct"])
        ground_truth = row["ground_truth"]

        for m in methods:
            if (rid, m) in done:
                continue
            score = float(row[m])
            jobs.append((rid, m, question, candidate_answer, score, correct, ground_truth))

    print(f"Queued {len(jobs)} DM calls (concurrency={concurrency}).")

    sem = asyncio.Semaphore(concurrency)
    pbar = tqdm(total=len(jobs), desc="DM calls (async)")

    async def run_one(job):
        rid, m, question, candidate_answer, score, correct, ground_truth = job
        async with sem:
            out = await call_decision_maker_async(question, candidate_answer, m, score)

        return {
            "id": rid,
            "method": m,
            "question": question,
            "candidate_answer": candidate_answer,
            "provided_score": float(score),
            "final_answer": out.final_answer,
            "final_confidence": float(out.final_confidence),
            "correct": int(correct),
            "ground_truth": ground_truth,
        }

    n_since_save = 0
    tasks = [asyncio.create_task(run_one(j)) for j in jobs]

    for fut in asyncio.as_completed(tasks):
        rec = await fut
        results.append(rec)
        done.add((rec["id"], rec["method"]))
        n_since_save += 1
        pbar.update(1)

        # if n_since_save >= save_every:
        #     pd.DataFrame(results).to_parquet(results_path, index=False)
        #     n_since_save = 0

    pbar.close()

    out_df = pd.DataFrame(results)
    out_df.to_parquet(results_path, index=False)
    return out_df

def run_dm_experiment(
    df: pd.DataFrame,
    methods: List[str] = METHODS,
    results_path: str = RESULTS_PATH,
    max_rows: Optional[int] = None,
    concurrency: int = DM_CONCURRENCY,
    save_every: int = SAVE_EVERY,
) -> pd.DataFrame:
    """
    Sync wrapper (so the rest of the notebook can keep calling run_dm_experiment(test_df)).
    Uses the already-running Jupyter event loop via nest_asyncio.
    """
    loop = asyncio.get_event_loop()
    return loop.run_until_complete(
        run_dm_experiment_async(
            df=df,
            methods=methods,
            results_path=results_path,
            max_rows=max_rows,
            concurrency=concurrency,
            save_every=save_every,
        )
    )


In [ ]:
def qa_correct(ans: str, gts) -> int:
    """TriviaQA correctness: substring match against aliases."""
    if ans is None:
        return 0
    # a = str(ans).replace("\\text{", "").lower()
    a = str(ans).split("\\text{")[-1].lower().replace("\\","")
    # gts should be list[str]
    for gt in gts:
        t = str(gt).lower()
        if (a in t) or (t in a):
            return 1
    return 0


def _get_method_stats(dm_df, method, dataset):

    method_df = dm_df[dm_df["method"] == method]

    print("[metrics] computing correctness...")
    if dataset in ["trivia_qa", "webq"]:

        correct = np.fromiter(
            (qa_correct(a, gts) for a, gts in zip(method_df["final_answer"], method_df["ground_truth"])),
            dtype=np.int64,
            count=len(method_df),
        )
        
    elif dataset == "sciq":

        correct = np.fromiter(
            (qa_correct(a, [gts]) for a, gts in zip(method_df["final_answer"], method_df["ground_truth"])),
            dtype=np.int64,
            count=len(method_df),
        )
        
    else:
        correct = (method_df["final_answer"].fillna("") == method_df["ground_truth"].fillna("")).astype(int).to_numpy(dtype=np.int64)

    return correct, method_df["final_confidence"]

In [ ]:
def run_exp(
    dataset,
    model,
    n=1000,
    k_train=100,
    temp_train=0.7,
    temp_test=0.6,
    prob_models=("ridge_clip", "split_isotonic_on_ridge", "split_isotonic_on_ridge_nrt"),   # <-- pass a list/tuple of the methods above
    include_verbal_conf=True,
    random_state=42,
    n_bins=12,
    embedding_text="question_response",
    drop_bad_rows=True,
    test_prop=0.6,
    verbose=False,
):
    """
    prob_models controls which embedding->prob models to include.
    Example:
      prob_models=["ridge_clip","ridge_logit","hgb_logit","mlp_logit","isotonic_on_ridge"]
    """

    set_seed(random_state)

    model_name = model.split("/")[-1]

    out_df, df_save_path = load_out_df(dataset, model_name, n, temp_train, k_train, temp_test, embedding_text, drop_bad_rows)

    # --------------------
    # Split
    # --------------------
    train_df, test_df = train_test_split(
        out_df,
        test_size=test_prop,
        random_state=random_state,
        shuffle=True,
    )

    X_train = stack_embeddings(train_df, "embeddings")
    y_train = train_df["consistency"].astype(np.float32).to_numpy()  # target in [0,1]

    X_test = stack_embeddings(test_df, "embeddings")

    # These are for evaluation (your core metrics are vs correctness)
    con_scores = test_df["consistency"].to_numpy(dtype=float)
    correct = test_df["correct"].to_numpy(dtype=int)

    logprobs = test_df["avg_logprobs"].to_numpy(dtype=float)
    ans_logprobs = test_df["ans_logprobs"].to_numpy(dtype=float)
    lp = np.exp(logprobs)
    alp = np.exp(ans_logprobs)

    if verbose:
        print(f"Results: (accuracy={correct.mean():.4f})")

    def metric_row(name, scores, correct, n_bins):
        return [
            name,
            get_ece1(scores, correct, n_bins=n_bins),
            get_ece2(scores, correct, n_bins=n_bins),
            get_mce(scores, correct, n_bins=n_bins),
            get_nll(scores, correct),
            brier_score_loss(correct, scores),
            roc_auc_score(correct, scores),
        ]
    
    
    score_sources = {
        "logprob": lp,
        "ans_logprob": alp,
        "oracle_sc": con_scores,
    }
    
    rows = [metric_row(name, scores, correct, n_bins) for name, scores in score_sources.items()]


    ########################
    # Ours
    ########################
    prob_models = list(prob_models) if prob_models is not None else []
    preds = fit_predict_prob_models(
        X_train=X_train,
        y_train_prob=y_train,
        X_test=X_test,
        methods=prob_models,
        random_state=random_state,
    )

    for method_name, pred_probs in preds.items():
        rows.append(metric_row(method_name, pred_probs, correct, n_bins))

    ########################
    # Verbal Confidence
    ########################
    vc_scores = None
    if include_verbal_conf:
        rows, vc_scores = maybe_add_verbal_conf_row(
            rows, 
            df_save_path, 
            test_df, 
            correct, 
            stem_idx=6
        )
    
    # --------------------
    # Pack + display
    # --------------------
    exp_df = pd.DataFrame(rows, columns=["Method", "ECE1", "ECE2", "MCE", "NLL", "Brier", "AUROC"])
    exp_df["Model"] = model
    exp_df["Dataset"] = dataset
    exp_df["Accuracy"] = float(correct.mean())
    exp_df = exp_df[["Model", "Dataset", "Method", "ECE1", "ECE2", "MCE", "NLL", "Brier", "AUROC", "Accuracy"]]
    
    display(exp_df)

    OUR_METHOD = "split_isotonic_on_ridge_nrt"

    print("\n\n")
    print("----"*20)
    print("\n\n")

    test_df["ours"] = preds[OUR_METHOD]
    test_df["vc"] = vc_scores
    test_df["lp"] = lp

    print("No. rows:", len(test_df))
    print(test_df.columns)
    print()
    display(test_df.head())

    dm_df = run_dm_experiment(test_df)  

    cal_model_name = model.split("/")[-1].replace("-", "_")
    test_df.to_csv(f"../outputs/sim_decisions/sim_{cal_model_name}_{MODEL_NAME}_{dataset}_{OUR_METHOD}_{random_state}_test_df.csv", index=False)
    dm_df.to_csv(f"../outputs/sim_decisions/sim_{cal_model_name}_{MODEL_NAME}_{dataset}_{OUR_METHOD}_{random_state}_dm_df.csv", index=False)

    display(dm_df.head())
    
    methods = ["vc", "lp", "ours"]
    
    rows = []
    columns = ["Method", "ECE1", "ECE2", "MCE", "NLL", "Brier", "AUROC", "DM_acc", "OG_acc"]
    
    for method in methods:
    
        # print(method)
    
        correct, pred_probs = _get_method_stats(dm_df, method, dataset)
        
        # pred_probs = method_confidence[method]
        # correct = method_correctness[method]
    
        og_correct = dm_df[dm_df["method"] == method]["correct"]
    
        # print(np.mean(pred_probs))
        # print(np.mean(correct))
        # print(np.mean(og_correct))
        
        rows.append([
            method,
            get_ece1(pred_probs, correct, n_bins=n_bins),
            get_ece2(pred_probs, correct, n_bins=n_bins),
            get_mce(pred_probs, correct, n_bins=n_bins),
            get_nll(pred_probs, correct),
            brier_score_loss(correct, pred_probs),
            roc_auc_score(correct, pred_probs),
            np.mean(correct),
            np.mean(og_correct)
        ])
    
    exp_df = pd.DataFrame(rows, columns=columns)

    display(
        exp_df[["Method", "ECE1", "ECE2", "MCE", "Brier", "AUROC", "DM_acc", "OG_acc"]]
    )
    
        
    return test_df, dm_df, exp_df
        


In [ ]:
models = [
    "Qwen/Qwen3-0.6B",
    "Qwen/Qwen3-1.7B", 
    "Qwen/Qwen3-4B-Thinking-2507", 
    "Qwen/Qwen3-8B",
    "Qwen/Qwen3-14B",
]
MODEL_NAME = "gpt-4o-nano"
dataset = "sciq"
random_state=10

for model in models:

    print("**--**--"*10)
    print("**--**--"*10)
    print(f"[status] running {model}")
    print("**--**--"*10)
    
    test_df, dm_df, exp_df = run_exp(
        dataset=dataset,
        model=model,
        random_state=random_state,
        test_prop=0.6
    )
    print("**--**--"*30)
    print("**--**--"*30)
    print();print()